In [1]:
import argparse, os, re, json, glob, sys
from typing import List, Dict, Any, Tuple

import dspy
from dspy import Signature, InputField, OutputField, Predict
from dspy.evaluate import SemanticF1

# --- Simple HTML loader and TF‑IDF retriever ---
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
import dspy
import warnings
import api_key
warnings.filterwarnings("ignore", category=UserWarning, module='transformers')

try:
    lm = dspy.LM('xai/grok-3-mini', api_key=api_key.grok_key)
    dspy.configure(lm=lm)
    print("LLM for evaluation configured successfully.")
except Exception as e:
    print(f"Could not configure LLM, evaluation might fail. Error: {e}")
    print("Please ensure you have a valid API key set for your chosen model.")

LLM for evaluation configured successfully.


In [18]:
def load_html_texts(sources_dir: str) -> List[str]:
    texts = []
    paths = glob.glob(os.path.join(sources_dir, "**", "*.html"), recursive=True)
    print(f"Found {len(paths)} HTML files in {sources_dir}.")
    
    # keep only 1/100 of the files
    sample_size = max(1, len(paths) // 100)
    paths = paths[:sample_size]
    print(f"Using {len(paths)} HTML files (1/100 of total).")
    
    for p in paths:
        try:
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                soup = BeautifulSoup(f.read(), "html.parser")
            t = soup.get_text(separator=" ", strip=True)
            t = re.sub(r"\s+", " ", t)
            if len(t) > 80:
                texts.append(t)
        except Exception:
            pass
    
    if not texts:
        raise RuntimeError("No HTML texts found. Check --sources_dir.")
    return texts


In [5]:
class MyRetriever:
    def __init__(self, texts: List[str], max_features: int = 60000):
        self.texts = texts
        self.vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1,2))
        self.matrix = self.vectorizer.fit_transform(texts)

    def search(self, query: str, k: int = 6) -> List[str]:
        qv = self.vectorizer.transform([query])
        sims = cosine_similarity(qv, self.matrix).ravel()
        idx = sims.argsort()[::-1][:k]
        return [self.texts[i] for i in idx]

In [42]:
# --- Dataset utils ---
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out

def first_turn_items_func(convs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    items = []
    for c in convs:
        if not c.get("qas"): continue
        q0 = c["qas"][0]["q"]
        a0 = c["qas"][0]["a"]
        items.append({"history": [], "question": q0, "gold": a0})
    return items

def all_turn_items(convs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    items = []
    for c in convs:
        hist: List[Tuple[str,str]] = []
        for qa in c.get("qas", []):
            q = qa["q"]; a = qa.get("a","")
            items.append({"history": hist.copy(), "question": q, "gold": a})
            hist = hist + [(q, a)]  # advance gold history
    return items

In [7]:
# --- DSPy Signatures ---
class HistSumSig(Signature):
    history = InputField(desc="List of (q,a) pairs so far, newest last.")
    summary = OutputField(desc="One-paragraph summary of the student's goals/interests.")

class NeedSig(Signature):
    history_summary = InputField()
    question = InputField()
    retrieved_context = InputField()
    need = OutputField(desc="A short sentence capturing the unspoken need.")

class CoopQuerySig(Signature):
    question = InputField()
    history_summary = InputField()
    pragmatic_need = InputField()
    coop_query = OutputField(desc="A refined query to fetch missing-but-helpful details.")

class FinalSig(Signature):
    history = InputField()
    question = InputField()
    history_summary = InputField()
    pragmatic_need = InputField()
    contexts = InputField()
    answer = OutputField(desc="Concise, cooperative, pragmatically-aware answer.")

In [8]:
# --- Pipeline module ---
class CooperativeDSPy(dspy.Module):
    def __init__(self, retriever: MyRetriever, k: int = 6):
        super().__init__()
        self.retriever = retriever
        self.k = k
        self.hist_sum = Predict(HistSumSig)
        self.need = Predict(NeedSig)
        self.coopq = Predict(CoopQuerySig)
        self.final = Predict(FinalSig)

    def _retrieve(self, q: str) -> str:
        ctxs = self.retriever.search(q, k=self.k)
        return "\n\n".join(ctxs)

    def forward(self, history: List[Tuple[str,str]], question: str) -> dspy.Prediction:
        main_ctx = self._retrieve(question)
        hs = self.hist_sum(history=history).summary
        need = self.need(history_summary=hs, question=question, retrieved_context=main_ctx).need
        cq = self.coopq(question=question, history_summary=hs, pragmatic_need=need).coop_query
        coop_ctx = self._retrieve(cq) if cq else ""
        merged = (main_ctx + "\n\n" + coop_ctx).strip()
        ans = self.final(history=history, question=question, history_summary=hs,
                         pragmatic_need=need, contexts=merged).answer
        return dspy.Prediction(answer=ans)


In [56]:
        # pred = program(history=it["history"], question=it["question"]).answer  # str
from types import SimpleNamespace
import dspy
from dspy.evaluate.auto_evaluation import SemanticF1  # auto-evaluator

def evaluate(program, items, max_n=None):
    judge = SemanticF1()  # returns a float F1 per example
    f1s = []

    N = len(items) if max_n is None else min(max_n, len(items))
    for i in range(N):
        print(f"Evaluating item {i+1}/{N}...")
        it = items[i]
        # run your program
        pred_text = program(history=it["history"], question=it["question"]).answer
        # build the objects judge expects
        example = dspy.Example(question=it["question"], response=it["gold"]).with_inputs("question")
        pred_obj = SimpleNamespace(response=pred_text)

        f1 = judge(example, pred_obj)  # float
        f1s.append(f1)

    avg_f1 = sum(f1s) / len(f1s) if f1s else 0.0
    return {"f1": avg_f1}

In [ ]:
data_dir = "../PragmatiCQA/data"
sources_dir = "../PragmatiCQA-sources"
split = "val"              # or "train", "test"
mode = "first_only"        # or "full"
topk = 3                   # how many passages to retrieve
max_eval = None            # or an integer to limit examples

In [19]:
# Build retriever once
print("[load] indexing HTML sources...", file=sys.stderr)
texts = load_html_texts(sources_dir)
retriever = MyRetriever(texts)

[load] indexing HTML sources...


Found 28570 HTML files in ../PragmatiCQA-sources.
Using 285 HTML files (1/100 of total).


In [20]:
# Load dataset
path = os.path.join(data_dir, f"{split}.jsonl")
if not os.path.isfile(path):
    print(f"ERROR: missing split file: {path}", file=sys.stderr); sys.exit(1)
convs = load_jsonl(path)

In [57]:
first_turn_items = first_turn_items_func(convs)
first_turn_items = first_turn_items[:len(first_turn_items)//3] 
print(f"[eval] {mode} examples: {len(first_turn_items)}", file=sys.stderr)

program = CooperativeDSPy(retriever, k=topk)
report = evaluate(program, first_turn_items, max_n=max_eval)

print(json.dumps(report, indent=2))

[eval] first_only examples: 59
2025/08/24 20:16:23 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:16:23 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/24 20:16:23 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:16:23 WARNING dspy.adapters.json_adapter: Failed to use structured output format, fallin

Evaluating item 1/59...
Evaluating item 2/59...


2025/08/24 20:16:32 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:16:32 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Evaluating item 3/59...


2025/08/24 20:16:38 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:16:38 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/24 20:16:38 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:16:38 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


...

Evaluating item 56/59...


2025/08/24 20:26:15 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:26:15 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/24 20:26:15 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:26:15 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Evaluating item 57/59...
Evaluating item 58/59...


2025/08/24 20:26:47 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:26:47 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/24 20:26:47 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 20:26:47 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Evaluating item 59/59...
{
  "f1": 0.3298922937428869
}


### first turn items:
### f1: 0.3298922937428869

In [63]:
# all_turn_items = all_turn_items(convs)
all_turn_items = all_turn_items[:len(all_turn_items)//4] 

print(f"[eval] {mode} examples: {len(all_turn_items)}", file=sys.stderr)

program = CooperativeDSPy(retriever, k=topk)
report = evaluate(program, all_turn_items, max_n=max_eval)

print(json.dumps(report, indent=2))


[eval] first_only examples: 19
2025/08/24 22:21:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 22:21:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/24 22:21:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 22:21:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, fallin

Evaluating item 1/19...
Evaluating item 2/19...
Evaluating item 3/19...
Evaluating item 4/19...
Evaluating item 5/19...
Evaluating item 6/19...
Evaluating item 7/19...
Evaluating item 8/19...
Evaluating item 9/19...
Evaluating item 10/19...


2025/08/24 22:21:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/24 22:21:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 22:21:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Evaluating item 11/19...
Evaluating item 12/19...


2025/08/24 22:22:25 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 22:22:25 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/24 22:23:13 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 22:23:13 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


...

Evaluating item 18/19...


2025/08/24 22:29:17 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 22:29:17 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Evaluating item 19/19...


2025/08/24 22:30:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.
2025/08/24 22:30:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


{
  "f1": 0.17117097787102442
}


## all turn 
## f1: 0.17117097787102442

gsdgdsggdsgdssssssssssss

In [1]:
import json

def load_pragmaticqa_first_questions(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            conversation = json.loads(line)
            if not conversation['qas']:
                continue
            
            first_qa = conversation['qas'][0]
            
            literal_context = " ".join([span['text'] for span in first_qa['a_meta'].get('literal_obj', [])])
            pragmatic_context = " ".join([span['text'] for span in first_qa['a_meta'].get('pragmatic_obj', [])])
            
            data.append({
                'question': first_qa['q'],
                'gold_answer': first_qa['a'],
                'literal_context': literal_context.strip(),
                'pragmatic_context': pragmatic_context.strip()
            })
    return data

val_data_path = '../PragmatiCQA/data/val.jsonl'
val_data = load_pragmaticqa_first_questions(val_data_path)

print(f"Loaded {len(val_data)} first-question examples from {val_data_path}")
print("\nExample data point:")
print(json.dumps(val_data[0], indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '../PragmatiCQA/data/val.jsonl'

In [ ]:
import os
from glob import glob
from bs4 import BeautifulSoup
from tqdm import tqdm

def read_corpus_from_sources(sources_path):
   
    search_pattern = os.path.join(sources_path, '*', '*.html')
    
    html_files = glob(search_pattern)
    
    if not html_files:
        raise FileNotFoundError(f"No HTML files were found using the pattern: {search_pattern}. Please verify the 'sources_path' is correct.")
        
    print(f"Found {len(html_files)} HTML files to process.")
    
    # Process each file and extract its text
    corpus_texts = []
    for file_path in tqdm(html_files, desc="Reading files"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f, 'html.parser')
                corpus_texts.append(soup.get_text())
        except Exception as e:
            print(f"Warning: Could not read file {file_path}. Error: {e}")
            
    return corpus_texts